In [1]:
import re
from langchain_community.document_loaders import TextLoader
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain_community.chat_models import ChatOllama
from langchain.chains import RetrievalQA
import numpy as np

# Clean biomedical text for RAG
def clean_research_text(text):
    text = re.sub(r'\[\d+(,\s*\d+)*\]', '', text)
    text = re.sub(r'\((\d{4}[a-z]?|[A-Z][a-z]+ et al\., \d{4})\)', '', text)
    text = re.sub(r'Figure\s?\d+[a-zA-Z]?', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Table\s?\d+[a-zA-Z]?', '', text, flags=re.IGNORECASE)
    text = re.split(r'\n\s*References\s*\n', text, flags=re.IGNORECASE)[0]
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\n.*(University|Institute|Hospital|Center|Centre|Dept\.|Department).*?\n', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

# Load and clean documents
def load_documents():
    loader = PyPDFDirectoryLoader(r"C:\Users\haris\OneDrive\Desktop\AI\Projects\Chatbot-RAG\data")
    docs = loader.load()
    for doc in docs:
        doc.page_content = clean_research_text(doc.page_content)
    return docs

docs = load_documents()
print(f"Loaded {len(docs)} documents.")
print(docs[0].page_content[:500])

# Text splitting
splitter = RecursiveCharacterTextSplitter(chunk_size=1600, chunk_overlap=400)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks.")
print(chunks[23].page_content[:300])

# Embedding and FAISS vector store
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
vectorstore.save_local("faiss_index")

# Test embedding vector
vector = embedding.embed_documents([chunks[0].page_content])
print(vector[0][:10])

# Load vector store
vectorstore = FAISS.load_local(
    r"C:\Users\haris\OneDrive\Desktop\AI\Projects\Chatbot-RAG\faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)

# Access raw FAISS vectors
faiss_index = vectorstore.index
vectors = faiss_index.reconstruct_n(0, faiss_index.ntotal)
print(np.array(vectors[0])[:10])

# Check document store
keys = list(vectorstore.docstore._dict.keys())
print(keys[0])
doc = vectorstore.docstore._dict[keys[0]]
print(doc.page_content[:300])

# LLM and QA
llm = ChatOllama(model="mistral")  # local Mistral
qa = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever(search_kwargs={"k": 4}))
response = qa.run("Who is Napoleon in this?")
print(response)


Loaded 867 documents.
Frontiers in Medicine 01 frontiersin.org
Advances in melanoma: epidemiology, diagnosis, and prognosis
Shayan Waseh 1 and Jason B. Lee 2*
of Dermatology, Thomas Jefferson University, Philadelphia, PA, United States
Unraveling the multidimensional complexities of melanoma has required concerted efforts by dedicated community of researchers and clinicians battling against this deadly form of skin cancer. Remarkable advances have been made in the realm of epidemiology, classification, diagnosis, and
Split into 2160 chunks.
interobserver reliability. In addition, the lack of standardized diagnostic terms for melanocytic neoplasms and disagreements about the fundamental nature of various melanocytic neoplasms have contributed to the confusion among clinicians and patients and the lack of standard management. The local a


C:\Users\haris\AppData\Local\Temp\ipykernel_11816\2330889510.py:44: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")



[-0.0484735444188118, -0.00011878545774379745, 0.015635274350643158, 0.03118092380464077, 0.04102334752678871, -0.03333719074726105, 0.030555495992302895, 0.04364295303821564, -0.0740414708852768, -0.0071295215748250484]
[-0.04847357 -0.00011877  0.01563523  0.03118091  0.04102336 -0.03333717
  0.03055551  0.04364295 -0.07404148 -0.00712952]
23852b55-ea3c-4c06-88ac-ae3f45280e8b
Frontiers in Medicine 01 frontiersin.org
Advances in melanoma: epidemiology, diagnosis, and prognosis
Shayan Waseh 1 and Jason B. Lee 2*
of Dermatology, Thomas Jefferson University, Philadelphia, PA, United States
Unraveling the multidimensional complexities of melanoma has required concerted effort


C:\Users\haris\AppData\Local\Temp\ipykernel_11816\2330889510.py:71: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="mistral")  # local Mistral
C:\Users\haris\AppData\Local\Temp\ipykernel_11816\2330889510.py:73: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa.run("Who is Napoleon in this?")


 Based on the provided context, there is no direct mention or reference to Napoleon in the given data. The text appears to contain health-related statistics about various countries and diseases such as kidney cancer, laryngeal cancer, leukaemia, lip cancer, liver cancer, lung cancer, lymphoma, melanoma of the skin, mesothelioma, chronic myeloid leukemia (CML), colon cancer, colorectal cancer, eye cancer, gynaecological cancers, and head and neck cancer. Napoleon is a historical figure who lived in the 19th century and was not mentioned in this context.
